In [1]:
import pandas as pd
import numpy as np

In [2]:
sail_cluster = pd.read_excel('Sail_Cluster.xlsx')
sail_cluster.head()

,adjusted_date,speed,mean_draft,sea_state,me_actual_steaming_time,ae_t_steaming,aux_running,blr_running,Cluster
0,2024-01-01,9.71,11.4,2,17.0,0,3,0,1
1,2024-01-01,10.00,11.4,2,1.0,0,3,0,1
2,2024-01-02,8.57,11.4,2,0.7,0,1,0,1
3,2024-01-02,9.50,11.4,2,22.0,0,2,0,1
4,2024-01-03,10.42,11.4,2,2.4,0,3,0,1


In [3]:
sail_cluster['Cluster'].value_counts()

Cluster
1    560
Name: count, dtype: int64

In [7]:
port_cluster = pd.read_excel('Port_Cluster.xlsx')
port_cluster.head()

,adjusted_date,mean_draft,ae_t_steaming,aux_running,blr_running,activity_time,Cluster
0,2024-01-01,11.40,0,3,0,5.6,1
1,2024-01-03,11.40,0,1,0,17.3,1
2,2024-01-04,10.55,0,3,0,5.5,1
3,2024-01-07,8.40,0,3,0,18.8,1
4,2024-01-08,8.40,0,1,0,10.5,1


In [9]:
port_cluster.shape

(193, 7)

In [11]:
port_cluster['Cluster'].value_counts()

Cluster
1    193
Name: count, dtype: int64

In [13]:
sail_cluster[sail_cluster['Cluster'] == 1]['me_actual_steaming_time'].sum()/24, sail_cluster[sail_cluster['Cluster'] == 2]['me_actual_steaming_time'].sum()/24, sail_cluster[sail_cluster['Cluster'] == 3]['me_actual_steaming_time'].sum()/24

(np.float64(188.9375), np.float64(0.0), np.float64(0.0))

In [15]:
import warnings
warnings.filterwarnings('ignore')

In [34]:
import pandas as pd

def get_random_sample_with_time_sum(df, time_column, target_sum):
    while True:
        # Shuffle the DataFrame
        df_shuffled = df.sample(frac=1).reset_index(drop=True)

        df_sample = pd.DataFrame(columns=df.columns)
        current_sum = 0

        # Iterate over the shuffled DataFrame
        for _, row in df_shuffled.iterrows():
            if current_sum + row[time_column] <= target_sum:
                df_sample = pd.concat([df_sample, pd.DataFrame([row])], ignore_index=True)  # Use concat to append the row
                current_sum += row[time_column]
                
            # Check if the exact target sum is reached
            if current_sum == target_sum:
                # Check for duplicates in the resulting sample
                duplicates = df_sample[df_sample.duplicated()]
                if not duplicates.empty:
                    print("Duplicates found in the sample:")
                    print(duplicates)
                else:
                    print("No duplicates in the sample.")
                return df_sample
        
        # If the exact sum isn't reached, repeat the process
        #print("No exact match found. Shuffling again.")

def find_subset_df(data, cluster_no, expected_days, activity):
    cluster_data = data[data['Cluster'] == cluster_no]
    target_sum = expected_days * 24
    
    if activity == 'SAIL':
        time_column = 'me_actual_steaming_time'
        return get_random_sample_with_time_sum(cluster_data, time_column, target_sum)
    
    elif activity == 'PORT':
        time_column = 'activity_time'
        return get_random_sample_with_time_sum(cluster_data, time_column, target_sum)
    
    elif activity == 'BOTH':
        sail_data = get_random_sample_with_time_sum(cluster_data, 'me_actual_steaming_time', target_sum)
        port_data = get_random_sample_with_time_sum(cluster_data, 'activity_time', target_sum)
        return sail_data, port_data




In [36]:
port_sample = find_subset_df(port_cluster, cluster_no=1, expected_days=21, activity='PORT')
port_sample

No duplicates in the sample.


,adjusted_date,mean_draft,ae_t_steaming,aux_running,blr_running,activity_time,Cluster
0,2024-07-06,10.60,0,1,0,1.5,1
1,2024-09-07,9.45,0,2,0,12.8,1
2,2024-08-22,6.90,0,3,0,12.1,1
3,2024-01-08,8.40,0,1,0,10.5,1
4,2024-06-03,10.30,0,3,0,24.0,1
5,2024-09-28,10.30,0,3,0,6.6,1
6,2024-07-14,9.50,0,1,0,15.7,1
7,2024-11-08,9.35,0,2,0,6.9,1
8,2024-06-25,10.20,0,3,0,7.5,1
9,2024-02-01,9.95,0,1,0,0.8,1


In [38]:
sail_sample = find_subset_df(sail_cluster, cluster_no=1, expected_days=44, activity='SAIL')
sail_sample

No duplicates in the sample.


,adjusted_date,speed,mean_draft,sea_state,me_actual_steaming_time,ae_t_steaming,aux_running,blr_running,Cluster
0,2024-01-04,10.71,10.55,2,1.4,0,3,0,1
1,2024-07-29,7.27,8.65,2,2.2,0,3,0,1
2,2024-01-07,10.00,10.55,2,1.9,0,3,0,1
3,2024-09-11,6.00,10.40,2,4.0,0,3,0,1
4,2024-01-25,10.67,9.50,3,1.5,0,1,0,1
...,...,...,...,...,...,...,...,...,...
137,2024-06-18,6.67,8.85,2,1.2,0,3,0,1
138,2024-09-10,3.08,10.40,2,2.6,0,3,0,1
139,2024-04-15,10.33,9.25,2,1.5,0,3,0,1
140,2024-04-05,9.03,9.25,4,3.1,0,3,0,1


In [40]:
port_sample.to_excel('Subset_PORT_DF_corrected.xlsx')

In [25]:
sail_sample.to_excel('Subset_SAIL_DF_corrected.xlsx')

# ORGINIAL